In [8]:
# Install required libraries
!pip install transformers datasets==2.19.0 seqeval torch accelerate -q

print("✅ Installation complete!")

✅ Installation complete!


In [9]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Verify GPU
print(f"🔥 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

🔥 GPU Available: True
GPU Device: Tesla T4


In [10]:
# Load CoNLL-2003 dataset
print("📥 Loading CoNLL-2003 dataset...")
# Load the parquet version which doesn't require a python script
dataset = load_dataset("eriktks/conll2003")

print("\n✅ Dataset loaded successfully!")
print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Test samples: {len(dataset['test'])}")

# Display first example
print("\n📊 First training example:")
print(dataset['train'][0])

📥 Loading CoNLL-2003 dataset...

✅ Dataset loaded successfully!
Train samples: 14041
Validation samples: 3250
Test samples: 3453

📊 First training example:
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [12]:
# Get label information
pos_tags = dataset['train'].features['pos_tags'].feature.names
chunk_tags = dataset['train'].features['chunk_tags'].feature.names

print("🏷️ POS Tags:")
for i, tag in enumerate(pos_tags):
    print(f"  {i}: {tag}")

print("\n🏷️ Chunk Tags:")
for i, tag in enumerate(chunk_tags):
    print(f"  {i}: {tag}")

# Sample sentence visualization
sample = dataset['train'][0]
print("\n📝 Sample Sentence:")
print(f"Tokens: {sample['tokens']}")
print(f"POS Tags: {[pos_tags[i] for i in sample['pos_tags']]}")
print(f"Chunk Tags: {[chunk_tags[i] for i in sample['chunk_tags']]}")

🏷️ POS Tags:
  0: "
  1: ''
  2: #
  3: $
  4: (
  5: )
  6: ,
  7: .
  8: :
  9: ``
  10: CC
  11: CD
  12: DT
  13: EX
  14: FW
  15: IN
  16: JJ
  17: JJR
  18: JJS
  19: LS
  20: MD
  21: NN
  22: NNP
  23: NNPS
  24: NNS
  25: NN|SYM
  26: PDT
  27: POS
  28: PRP
  29: PRP$
  30: RB
  31: RBR
  32: RBS
  33: RP
  34: SYM
  35: TO
  36: UH
  37: VB
  38: VBD
  39: VBG
  40: VBN
  41: VBP
  42: VBZ
  43: WDT
  44: WP
  45: WP$
  46: WRB

🏷️ Chunk Tags:
  0: O
  1: B-ADJP
  2: I-ADJP
  3: B-ADVP
  4: I-ADVP
  5: B-CONJP
  6: I-CONJP
  7: B-INTJ
  8: I-INTJ
  9: B-LST
  10: I-LST
  11: B-NP
  12: I-NP
  13: B-PP
  14: I-PP
  15: B-PRT
  16: I-PRT
  17: B-SBAR
  18: I-SBAR
  19: B-UCP
  20: I-UCP
  21: B-VP
  22: I-VP

📝 Sample Sentence:
Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
POS Tags: ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.']
Chunk Tags: ['B-NP', 'B-VP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'O']


In [13]:
# Configuration for POS Tagging
TASK = "POS"
label_column = "pos_tags"
label_list = dataset['train'].features[label_column].feature.names

print(f"🎯 Task: {TASK} Tagging")
print(f"📊 Number of labels: {len(label_list)}")
print(f"🏷️ Labels: {label_list}")

# Create label mappings
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

🎯 Task: POS Tagging
📊 Number of labels: 47
🏷️ Labels: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']


In [15]:
# Load DistilBERT tokenizer (faster than BERT)
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print(f"✅ Loaded tokenizer: {model_checkpoint}")
print(f"Vocab size: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Loaded tokenizer: distilbert-base-uncased
Vocab size: 30522


In [16]:
def tokenize_and_align_labels(examples):
    """
    Tokenize inputs and align labels with subword tokens
    """
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding=False,
        max_length=512
    )

    labels = []
    for i, label in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            # Special tokens get -100 (ignored in loss)
            if word_idx is None:
                label_ids.append(-100)
            # Only label first token of each word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Subword tokens get -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Test tokenization on one example
print("🧪 Testing tokenization...")
sample = dataset['train'][0]
print(f"Original tokens: {sample['tokens'][:5]}")
print(f"Original labels: {sample[label_column][:5]}")

tokenized = tokenize_and_align_labels(dataset['train'][:1])
print(f"\nTokenized input_ids length: {len(tokenized['input_ids'][0])}")
print(f"Labels length: {len(tokenized['labels'][0])}")
print("✅ Tokenization successful!")

🧪 Testing tokenization...
Original tokens: ['EU', 'rejects', 'German', 'call', 'to']
Original labels: [22, 42, 16, 21, 35]

Tokenized input_ids length: 11
Labels length: 11
✅ Tokenization successful!


In [17]:
# Apply tokenization to all splits
print("⚙️ Preprocessing dataset...")
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing"
)

print("✅ Preprocessing complete!")
print(f"Train size: {len(tokenized_datasets['train'])}")
print(f"Validation size: {len(tokenized_datasets['validation'])}")
print(f"Test size: {len(tokenized_datasets['test'])}")

⚙️ Preprocessing dataset...


Tokenizing:   0%|          | 0/14041 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3250 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3453 [00:00<?, ? examples/s]

✅ Preprocessing complete!
Train size: 14041
Validation size: 3250
Test size: 3453


In [18]:
# Load pre-trained model for token classification
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Model loaded: {model_checkpoint}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Number of labels: {model.num_labels}")

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"🔥 Model on: {device}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded: distilbert-base-uncased
Number of parameters: 66,399,023
Number of labels: 47
🔥 Model on: cuda


In [19]:
# Data collator for dynamic padding
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

print("✅ Data collator created")

✅ Data collator created


In [20]:
def compute_metrics(eval_pred):
    """
    Compute precision, recall, F1 using seqeval
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Calculate metrics
    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

print("✅ Metrics function defined")

✅ Metrics function defined


In [22]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"./pos-tagging-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    report_to="none"
)

print("✅ Training arguments configured:")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Epochs: {training_args.num_train_epochs}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Training arguments configured:
  - Learning rate: 2e-05
  - Batch size: 16
  - Epochs: 3


In [24]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer initialized")

✅ Trainer initialized


In [25]:
# Start training
print("🚀 Starting training...")
print("="*50)

train_result = trainer.train()

print("\n" + "="*50)
print("✅ Training complete!")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Training samples/second: {train_result.metrics['train_samples_per_second']:.2f}")

🚀 Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.241282,0.257766,0.911113,0.910251,0.910682
2,0.187372,0.227815,0.918325,0.914982,0.916650
3,0.159851,0.220313,0.920868,0.918767,0.919816


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Training complete!
Training time: 326.64 seconds
Training samples/second: 128.96


In [26]:
# Evaluate on test set
print("📊 Evaluating on test set...")
eval_results = trainer.evaluate(tokenized_datasets["test"])

print("\n" + "="*50)
print("📈 EVALUATION RESULTS (POS TAGGING)")
print("="*50)
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall:    {eval_results['eval_recall']:.4f}")
print(f"F1 Score:  {eval_results['eval_f1']:.4f}")
print("="*50)

📊 Evaluating on test set...



📈 EVALUATION RESULTS (POS TAGGING)
Precision: 0.9117
Recall:    0.9080
F1 Score:  0.9098


In [27]:
# Get detailed predictions
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

# Convert to label names
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

# Print classification report
print("📋 DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(true_labels, true_predictions))

📋 DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

           '       0.50      0.50      0.50        14
           B       0.88      0.83      0.86      1625
          BD       0.94      0.95      0.95      1687
          BG       0.91      0.89      0.90       480
          BN       0.87      0.87      0.87       816
          BP       0.88      0.85      0.87       331
          BR       0.78      0.42      0.55        43
          BS       0.00      0.00      0.00         9
          BZ       0.92      0.84      0.88       501
           C       1.00      0.99      1.00       765
           D       0.95      0.98      0.97      3447
          DT       0.96      0.89      0.92       115
           H       0.00      0.00      0.00         7
           J       0.83      0.79      0.81      2214
          JR       0.73      0.87      0.80        92
          JS       0.84      0.91      0.87        56
           N       0.89      0.89      0.89     

In [29]:
def predict_pos_tags(sentence):
    """
    Predict POS tags for a custom sentence
    """
    # Tokenize
    tokens = sentence.split()
    tokenized_inputs = tokenizer(tokens,
                               is_split_into_words=True,
                               return_tensors="pt",
                               truncation=True,
                               padding=True)

    # Align predictions with original tokens before moving to device
    word_ids = tokenized_inputs.word_ids(batch_index=0)

    # Move to device
    inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}

    # Predict
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predictions
    predictions = torch.argmax(outputs.logits, dim=2)

    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            predicted_labels.append(id2label[predictions[0][idx].item()])
        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

# Test inference
print("🧪 INFERENCE TEST - POS TAGGING")
print("="*70)

test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "Apple released a new iPhone yesterday"
]

for sentence in test_sentences:
    print(f"\n📝 Input: {sentence}")
    results = predict_pos_tags(sentence)
    print("🏷️ POS Tags:")
    for token, tag in results:
        print(f"  {token:15} → {tag}")
    print("-"*70)

🧪 INFERENCE TEST - POS TAGGING

📝 Input: John works at Google in California
🏷️ POS Tags:
  John            → NNP
  works           → VBZ
  at              → IN
  Google          → NNP
  in              → IN
  California      → NNP
----------------------------------------------------------------------

📝 Input: The quick brown fox jumps over the lazy dog
🏷️ POS Tags:
  The             → DT
  quick           → JJ
  brown           → NNP
  fox             → NNP
  jumps           → VBZ
  over            → IN
  the             → DT
  lazy            → JJ
  dog             → NN
----------------------------------------------------------------------

📝 Input: Apple released a new iPhone yesterday
🏷️ POS Tags:
  Apple           → NNP
  released        → VBD
  a               → DT
  new             → JJ
  iPhone          → NNP
  yesterday       → NN
----------------------------------------------------------------------


In [30]:
# Configuration for Chunking
TASK = "CHUNK"
label_column = "chunk_tags"
chunk_label_list = dataset['train'].features[label_column].feature.names

print(f"🎯 Task: Chunking")
print(f"📊 Number of labels: {len(chunk_label_list)}")
print(f"🏷️ Labels: {chunk_label_list}")

# Create label mappings
chunk_label2id = {label: i for i, label in enumerate(chunk_label_list)}
chunk_id2label = {i: label for i, label in enumerate(chunk_label_list)}

🎯 Task: Chunking
📊 Number of labels: 23
🏷️ Labels: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [31]:
def tokenize_and_align_labels_chunking(examples):
    """
    Tokenize inputs and align chunk labels
    """
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding=False,
        max_length=512
    )

    labels = []
    for i, label in enumerate(examples['chunk_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Preprocess for chunking
print("⚙️ Preprocessing for chunking...")
chunk_tokenized_datasets = dataset.map(
    tokenize_and_align_labels_chunking,
    batched=True,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing for chunking"
)
print("✅ Chunking preprocessing complete!")

⚙️ Preprocessing for chunking...


Tokenizing for chunking:   0%|          | 0/14041 [00:00<?, ? examples/s]

Tokenizing for chunking:   0%|          | 0/3250 [00:00<?, ? examples/s]

Tokenizing for chunking:   0%|          | 0/3453 [00:00<?, ? examples/s]

✅ Chunking preprocessing complete!


In [32]:
# Load model for chunking
chunk_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(chunk_label_list),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

chunk_model.to(device)
print(f"✅ Chunking model loaded and moved to {device}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Chunking model loaded and moved to cuda


In [36]:
def compute_metrics_chunking(eval_pred):
    """
    Compute metrics for chunking
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [chunk_label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [chunk_label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

# 🧪 TEST SECTION: Add this to see a printout immediately
print("Testing compute_metrics_chunking...")

# 1. Create fake predictions (batch_size=1, seq_len=3, num_labels=23)
# Let's pretend the model is 100% sure about one tag
fake_preds = np.zeros((1, 3, 23))
fake_preds[0, 1, 11] = 10  # Index 11 is usually 'B-NP'

# 2. Create fake labels (-100 is ignored, 11 is the target)
fake_labels = np.array([[-100, 11, -100]])

# 3. Run the function
results = compute_metrics_chunking((fake_preds, fake_labels))

# 4. Print the results
print("-" * 30)
print(f"✅ Function Check Successful!")
print(f"Precision: {results['precision']:.2f}")
print(f"Recall:    {results['recall']:.2f}")
print(f"F1 Score:  {results['f1']:.2f}")
print("-" * 30)

Testing compute_metrics_chunking...
------------------------------
✅ Function Check Successful!
Precision: 1.00
Recall:    1.00
F1 Score:  1.00
------------------------------


In [39]:
# Training arguments for chunking
chunk_training_args = TrainingArguments(
    output_dir="./chunking-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    push_to_hub=False,
    report_to="none"
)

# Initialize trainer
chunk_trainer = Trainer(
    model=chunk_model,
    args=chunk_training_args,
    train_dataset=chunk_tokenized_datasets["train"],
    eval_dataset=chunk_tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_chunking,
)

# Train
print("🚀 Starting chunking model training...")
chunk_train_result = chunk_trainer.train()
print("✅ Chunking training complete!")

🚀 Starting chunking model training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.199872,0.203215,0.907470,0.899678,0.903557
2,0.162431,0.181407,0.912989,0.906710,0.909839
3,0.129312,0.174590,0.915711,0.911761,0.913732


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


✅ Chunking training complete!


In [40]:
# Evaluate chunking model
print("📊 Evaluating chunking model on test set...")
chunk_eval_results = chunk_trainer.evaluate(chunk_tokenized_datasets["test"])

print("\n" + "="*50)
print("📈 EVALUATION RESULTS (CHUNKING)")
print("="*50)
print(f"Precision: {chunk_eval_results['eval_precision']:.4f}")
print(f"Recall:    {chunk_eval_results['eval_recall']:.4f}")
print(f"F1 Score:  {chunk_eval_results['eval_f1']:.4f}")
print("="*50)

📊 Evaluating chunking model on test set...



📈 EVALUATION RESULTS (CHUNKING)
Precision: 0.9055
Recall:    0.8972
F1 Score:  0.9013


In [42]:
def predict_chunks(sentence):
    """
    Predict chunk tags for a custom sentence
    """
    tokens = sentence.split()
    tokenized_inputs = tokenizer(tokens,
                       is_split_into_words=True,
                       return_tensors="pt",
                       truncation=True,
                       padding=True)

    # Capture word_ids before moving to device
    word_ids = tokenized_inputs.word_ids(batch_index=0)

    inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}

    with torch.no_grad():
        outputs = chunk_model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)

    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx != previous_word_idx:
            predicted_labels.append(chunk_id2label[predictions[0][idx].item()])
        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

# Test chunking
print("🧪 INFERENCE TEST - CHUNKING")
print("="*70)

for sentence in test_sentences:
    print(f"\n📝 Input: {sentence}")
    results = predict_chunks(sentence)
    print("📦 Chunk Tags:")
    for token, tag in results:
        print(f"  {token:15} → {tag}")
    print("-"*70)

🧪 INFERENCE TEST - CHUNKING

📝 Input: John works at Google in California
📦 Chunk Tags:
  John            → B-NP
  works           → B-VP
  at              → B-PP
  Google          → B-NP
  in              → B-PP
  California      → B-NP
----------------------------------------------------------------------

📝 Input: The quick brown fox jumps over the lazy dog
📦 Chunk Tags:
  The             → B-NP
  quick           → I-NP
  brown           → I-NP
  fox             → I-NP
  jumps           → B-VP
  over            → B-PP
  the             → B-NP
  lazy            → I-NP
  dog             → I-NP
----------------------------------------------------------------------

📝 Input: Apple released a new iPhone yesterday
📦 Chunk Tags:
  Apple           → B-NP
  released        → B-VP
  a               → B-NP
  new             → I-NP
  iPhone          → I-NP
  yesterday       → B-NP
----------------------------------------------------------------------


In [43]:
print("🔍 COMPARISON: POS TAGGING vs CHUNKING")
print("="*80)

test_sentence = "John works at Google in California"
print(f"📝 Test Sentence: {test_sentence}\n")

# POS Tagging
pos_results = predict_pos_tags(test_sentence)
print("1️⃣ POS TAGGING (Grammar-level):")
for token, tag in pos_results:
    print(f"   {token:15} → {tag}")

print()

# Chunking
chunk_results = predict_chunks(test_sentence)
print("2️⃣ CHUNKING (Phrase-level):")
for token, tag in chunk_results:
    print(f"   {token:15} → {tag}")

print("\n" + "="*80)
print("📊 PERFORMANCE COMPARISON:")
print(f"POS Tagging F1:  {eval_results['eval_f1']:.4f}")
print(f"Chunking F1:     {chunk_eval_results['eval_f1']:.4f}")
print("="*80)

🔍 COMPARISON: POS TAGGING vs CHUNKING
📝 Test Sentence: John works at Google in California

1️⃣ POS TAGGING (Grammar-level):
   John            → NNP
   works           → VBZ
   at              → IN
   Google          → NNP
   in              → IN
   California      → NNP

2️⃣ CHUNKING (Phrase-level):
   John            → B-NP
   works           → B-VP
   at              → B-PP
   Google          → B-NP
   in              → B-PP
   California      → B-NP

📊 PERFORMANCE COMPARISON:
POS Tagging F1:  0.9098
Chunking F1:     0.9013


In [44]:
# Save POS Tagging model
model.save_pretrained("./saved_pos_model")
tokenizer.save_pretrained("./saved_pos_model")
print("✅ POS Tagging model saved")

# Save Chunking model
chunk_model.save_pretrained("./saved_chunk_model")
tokenizer.save_pretrained("./saved_chunk_model")
print("✅ Chunking model saved")

# Download models (optional)
from google.colab import files
!zip -r pos_model.zip ./saved_pos_model
!zip -r chunk_model.zip ./saved_chunk_model
# files.download('pos_model.zip')
# files.download('chunk_model.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ POS Tagging model saved


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Chunking model saved
  adding: saved_pos_model/ (stored 0%)
  adding: saved_pos_model/model.safetensors (deflated 8%)
  adding: saved_pos_model/config.json (deflated 61%)
  adding: saved_pos_model/tokenizer.json (deflated 71%)
  adding: saved_pos_model/tokenizer_config.json (deflated 42%)
  adding: saved_chunk_model/ (stored 0%)
  adding: saved_chunk_model/model.safetensors (deflated 8%)
  adding: saved_chunk_model/config.json (deflated 61%)
  adding: saved_chunk_model/tokenizer.json (deflated 71%)
  adding: saved_chunk_model/tokenizer_config.json (deflated 42%)


In [47]:
print("""
📝 REPORT: POS TAGGING vs CHUNKING
====================================

1. TASK DIFFERENCES:

   POS Tagging:
   - Grammar-level token classification
   - Assigns grammatical categories (NOUN, VERB, ADJ, etc.)
   - Fine-grained linguistic analysis
   - 47 different tags in CoNLL-2003

   Chunking:
   - Phrase-level grouping
   - Identifies syntactic units (NP, VP, PP, etc.)
   - Uses IOB format (B-NP, I-NP, O, etc.)
   - 23 different chunk tags in CoNLL-2003

2. CHALLENGES FACED:

   ✓ Subword Tokenization:
     - BERT tokenizer splits words into subwords
     - Had to align labels correctly using word_ids()
     - Subword tokens labeled with -100 to ignore in loss

   ✓ Special Tokens:
     - [CLS] and [SEP] tokens need -100 labels
     - Handled in tokenize_and_align_labels function

   ✓ Label Alignment:
     - Only first subword gets the label
     - Subsequent subwords of same word get -100

3. OBSERVATIONS:

   Model Performance:
   - Both models achieved >90% F1 score
   - DistilBERT is faster than BERT with minimal performance drop
   - 3 epochs sufficient for convergence

   Inference Quality:
   - POS tagging more granular
   - Chunking better for extracting phrases
   - Both useful for different NLP tasks

4. USE CASES:

   POS Tagging: Text-to-speech, lemmatization, syntax analysis
   Chunking: Information extraction, entity recognition, parsing

""")


📝 REPORT: POS TAGGING vs CHUNKING

1. TASK DIFFERENCES:
   
   POS Tagging:
   - Grammar-level token classification
   - Assigns grammatical categories (NOUN, VERB, ADJ, etc.)
   - Fine-grained linguistic analysis
   - 47 different tags in CoNLL-2003
   
   Chunking:
   - Phrase-level grouping
   - Identifies syntactic units (NP, VP, PP, etc.)
   - Uses IOB format (B-NP, I-NP, O, etc.)
   - 23 different chunk tags in CoNLL-2003

2. CHALLENGES FACED:
   
   ✓ Subword Tokenization:
     - BERT tokenizer splits words into subwords
     - Had to align labels correctly using word_ids()
     - Subword tokens labeled with -100 to ignore in loss
   
   ✓ Special Tokens:
     - [CLS] and [SEP] tokens need -100 labels
     - Handled in tokenize_and_align_labels function
   
   ✓ Label Alignment:
     - Only first subword gets the label
     - Subsequent subwords of same word get -100

3. OBSERVATIONS:
   
   Model Performance:
   - Both models achieved >90% F1 score
   - DistilBERT is faster th